# Exploratory analyses

Sentiment during the writing process, post-publication references, and associations with correspondents.

In [ ]:
from shared import paths
from IPython.display import display


## Prepare annotation tables

In [ ]:
from annotations.tables import prepare_tables, as_list
from shared.labels import CATEGORY_INDEX
from shared.text import DIVIDER, ABBREVIATIONS, protect_abbreviations, restore_abbreviations
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import re

paths.prepare_directories()
tables = prepare_tables()
letters = tables['letters']
reviewed_letters = tables['reviewed_letters']
letters_df = tables['letters_df']
sentences_df = tables['sentences_df']
traces_df = tables['traces_df']
annotations_df = tables['annotations_df']
sentence_annotation_df = tables['sentence_annotation_df']
all_sentences = tables['all_sentences']


## Writing and sentiment

In [ ]:
from pathlib import Path
from collections import Counter
import re
import pandas as pd
from transformers import pipeline

TARGET_CATEGORY = 'Writing Process and Creative Insight'
TARGET_SUBCATEGORY = 'mood_and_psychological_states'
TARGET_TEMPORAL_PERSPECTIVE = 'current_work'
SENTIMENT_MODEL = 'siebert/sentiment-roberta-large-english'
output_dir = Path(str(paths.FIGURES_DIR))
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / 'mood_psychological_states_work_in_progress_timeline.png'
publication_periods = {
    'pickwick_papers': (1836, 1837),
    'oliver_twist': (1837, 1839),
    'nicholas_nickleby': (1838, 1839),
    'old_curiosity_shop': (1840, 1841),
    'master_humphreys_clock': (1840, 1841),
    'barnaby_rudge': (1841, 1841),
    'the_chimes': (1844, 1844),
    'martin_chuzzlewit': (1843, 1844),
    'the_battle_of_life': (1846, 1846),
    'dombey_and_son': (1846, 1848),
    'the_haunted_man_and_the_ghosts_bargain': (1848, 1848),
    'david_copperfield': (1849, 1850),
    'bleak_house': (1852, 1853),
    'little_dorrit': (1855, 1857),
    'great_expectations': (1860, 1861),
    'our_mutual_friend': (1864, 1865),
}
target_label_map = {
    'pickwick_papers': 'The Pickwick Papers',
    'oliver_twist': 'Oliver Twist',
    'nicholas_nickleby': 'Nicholas Nickleby',
    'old_curiosity_shop': 'The Old Curiosity Shop',
    'master_humphreys_clock': "Master Humphrey's Clock",
    'barnaby_rudge': 'Barnaby Rudge',
    'the_chimes': 'The Chimes',
    'martin_chuzzlewit': 'Martin Chuzzlewit',
    'the_battle_of_life': 'The Battle of Life',
    'dombey_and_son': 'Dombey and Son',
    'the_haunted_man_and_the_ghosts_bargain': 'The Haunted Man',
    'david_copperfield': 'David Copperfield',
    'bleak_house': 'Bleak House',
    'little_dorrit': 'Little Dorrit',
    'great_expectations': 'Great Expectations',
    'our_mutual_friend': 'Our Mutual Friend',
}
from annotations.tables import as_list


def contains_value(values, target):
    return target in as_list(values)


def extract_year(value):
    if value is None:
        return None
    if isinstance(value, int) and 1800 <= value <= 1899:
        return value
    match = re.search('\\b(18[0-9]{2})\\b', str(value))
    if match:
        return int(match.group(1))
    return None


def format_target(value):
    value = str(value).strip()
    return target_label_map.get(value, value.replace('_', ' ').title())


def clean_sentence(text):
    text = str(text)
    text = re.sub('\\s+', ' ', text)
    text = text.replace('**', '')
    return text.strip()
letter_meta_df = pd.DataFrame(
    {
        'letter_id': [letter.id for letter in reviewed_letters],
        'year': [extract_year(letter.year) for letter in reviewed_letters],
        'recipient': [getattr(letter, 'norm_recipient', None) for letter in reviewed_letters],
    },
)
df = sentences_df.copy()
df = df.merge(letter_meta_df, on='letter_id', how='left')
mask = df['has_reference'] & df['categories'].apply(lambda values: contains_value(values, TARGET_CATEGORY)) & df['subcategories'].apply(lambda values: contains_value(values, TARGET_SUBCATEGORY)) & df['temporal_perspectives'].apply(
    lambda values: contains_value(values, TARGET_TEMPORAL_PERSPECTIVE),
)
target_df = df[mask].copy()
target_df = target_df[target_df['year'].notna()].copy()
target_df['year'] = target_df['year'].astype(int)
if target_df.empty:
    raise ValueError(
        'No matching rows found. Check the category, subcategory, temporal perspective, or the availability of letter.year.',
    )
target_counter = Counter()
for values in target_df['target_novels']:
    for target in as_list(values):
        if target:
            target_counter[target] += 1
print('\nMood and Psychological States in references to literary work in progress')
print(f'Matching sentence-level references: {len(target_df):,}')
print(f"Letters represented: {target_df['letter_id'].nunique():,}")
print(f"Year range: {target_df['year'].min()}–{target_df['year'].max()}")
print(f'Target-work assignments in filtered references: {sum(target_counter.values()):,}')
print('\nTarget works represented:')
for target, count in target_counter.most_common():
    print(f'{format_target(target)}: {count}')
print('\nTarget works by year:')
for year, year_df in target_df.groupby('year'):
    year_counter = Counter()
    for values in year_df['target_novels']:
        for target in as_list(values):
            if target:
                year_counter[target] += 1
    target_string = '; '.join(
        (f'{format_target(target)} ({count})' for target, count in year_counter.most_common()),
    )
    print(f'{year}: {target_string}')
print('\nRunning sentiment analysis...')
print(f'Model: {SENTIMENT_MODEL}')
sentiment_pipe = pipeline(
    'sentiment-analysis',
    model=SENTIMENT_MODEL,
    tokenizer=SENTIMENT_MODEL,
    truncation=True,
    device=-1,
)
sentiment_texts = target_df['sentence_text'].apply(clean_sentence).tolist()
sentiment_results = sentiment_pipe(sentiment_texts, batch_size=16)
target_df['sentiment_label'] = [result['label'].lower() for result in sentiment_results]
target_df['sentiment_score'] = [result['score'] for result in sentiment_results]
sentiment_counts = target_df['sentiment_label'].value_counts()
sentiment_shares = target_df['sentiment_label'].value_counts(normalize=True) * 100
print('\nSentiment distribution:')
for label, count in sentiment_counts.items():
    print(f'{label}: {count} ({sentiment_shares[label]:.1f}%)')
print('\nMean model confidence by sentiment:')
print(target_df.groupby('sentiment_label')['sentiment_score'].mean().round(3).to_string())
sentiment_by_year = target_df.groupby(['year', 'sentiment_label']).size().unstack(fill_value=0).sort_index()
for label in ['negative', 'positive']:
    if label not in sentiment_by_year.columns:
        sentiment_by_year[label] = 0
sentiment_by_year = sentiment_by_year[['negative', 'positive']]
sentiment_by_year['total'] = sentiment_by_year['negative'] + sentiment_by_year['positive']
sentiment_by_year['negative_share'] = (sentiment_by_year['negative'] / sentiment_by_year['total'] * 100).round(1)
sentiment_by_year['positive_share'] = (sentiment_by_year['positive'] / sentiment_by_year['total'] * 100).round(1)
print('\nSentiment by year:')
print(sentiment_by_year.to_string())
target_sentiment_rows = []
for row in target_df.itertuples(index=False):
    for target in as_list(row.target_novels):
        if not target:
            continue
        target_sentiment_rows.append(
            {
                'target_work': format_target(target),
                'sentiment_label': row.sentiment_label,
                'sentiment_score': row.sentiment_score,
                'year': row.year,
                'letter_id': row.letter_id,
            },
        )
target_sentiment_df = pd.DataFrame(target_sentiment_rows)
target_sentiment_summary = target_sentiment_df.groupby(['target_work', 'sentiment_label']).size().unstack(fill_value=0)
for label in ['negative', 'positive']:
    if label not in target_sentiment_summary.columns:
        target_sentiment_summary[label] = 0
target_sentiment_summary = target_sentiment_summary[['negative', 'positive']]
target_sentiment_summary['total'] = target_sentiment_summary['negative'] + target_sentiment_summary['positive']
target_sentiment_summary['negative_share'] = (target_sentiment_summary['negative'] / target_sentiment_summary['total'] * 100).round(1)
target_sentiment_summary['positive_share'] = (target_sentiment_summary['positive'] / target_sentiment_summary['total'] * 100).round(1)
target_sentiment_summary = target_sentiment_summary.sort_values(['total', 'negative'], ascending=False)
print('\nSentiment by target work:')
print(target_sentiment_summary.to_string())
print('\nStrongest negative examples:')
negative_examples = target_df[target_df['sentiment_label'] == 'negative'].sort_values(
    'sentiment_score',
    ascending=False,
).head(
    10,
)
for row in negative_examples.itertuples(index=False):
    targets = ', '.join((format_target(target) for target in as_list(row.target_novels)))
    sentence = '[LETTER TEXT]'
    print(
        f'\n{row.year} | {row.letter_id} | {targets} | score={row.sentiment_score:.3f}\n{sentence}',
    )
print('\nStrongest positive examples:')
positive_examples = target_df[target_df['sentiment_label'] == 'positive'].sort_values(
    'sentiment_score',
    ascending=False,
).head(
    10,
)
for row in positive_examples.itertuples(index=False):
    targets = ', '.join((format_target(target) for target in as_list(row.target_novels)))
    sentence = '[LETTER TEXT]'
    print(
        f'\n{row.year} | {row.letter_id} | {targets} | score={row.sentiment_score:.3f}\n{sentence}',
    )
yearly_counts = target_df.groupby('year').size().reset_index(name='references')
period_targets = [target for target, _ in target_counter.most_common() if target in publication_periods]
period_targets = sorted(
    period_targets,
    key=lambda target: (publication_periods[target][0], publication_periods[target][1], format_target(target)),
)
x_min = min(target_df['year'].min(), min((publication_periods[target][0] for target in period_targets))) - 1
x_max = max(target_df['year'].max(), max((publication_periods[target][1] for target in period_targets))) + 1
full_year_range = pd.DataFrame({'year': range(int(x_min), int(x_max) + 1)})
plot_df = full_year_range.merge(yearly_counts, on='year', how='left').fillna({'references': 0})
plot_df['references'] = plot_df['references'].astype(int)
print('\nPrepared plot data.')
print('Sentiment predictions are ready for the timeline below.')


## Sentiment timeline

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

label_fs = 16
tick_fs = 14
value_fs = 13
legend_fs = 13
period_row_gap = 1.35
period_bar_height = 0.58
negative_color = '#9E4A4A'
positive_color = '#4F7F5F'
period_y_positions = {target: i * period_row_gap for i, target in enumerate(period_targets)}
fig_height = max(8.6, 5.4 + 0.42 * len(period_targets))
sentiment_plot_df = target_df.groupby(['year', 'sentiment_label']).size().unstack(fill_value=0).reset_index()
for label in ['negative', 'positive']:
    if label not in sentiment_plot_df.columns:
        sentiment_plot_df[label] = 0
sentiment_plot_df = plot_df[['year']].merge(sentiment_plot_df[['year', 'negative', 'positive']], on='year', how='left').fillna(
    {'negative': 0, 'positive': 0},
)
sentiment_plot_df['negative'] = sentiment_plot_df['negative'].astype(int)
sentiment_plot_df['positive'] = sentiment_plot_df['positive'].astype(int)
sentiment_plot_df['total'] = sentiment_plot_df['negative'] + sentiment_plot_df['positive']
fig, (ax_counts, ax_periods) = plt.subplots(
    2,
    1,
    figsize=(13.5, fig_height),
    sharex=True,
    gridspec_kw={'height_ratios': [2.6, 2.7], 'hspace': 0.08},
)
ax_counts.bar(
    sentiment_plot_df['year'],
    sentiment_plot_df['positive'],
    color=positive_color,
    edgecolor='0.55',
    width=0.82,
    label='Positive Sentiment',
)
ax_counts.bar(
    sentiment_plot_df['year'],
    sentiment_plot_df['negative'],
    bottom=sentiment_plot_df['positive'],
    color=negative_color,
    edgecolor='0.55',
    width=0.82,
    label='Negative Sentiment',
)
max_count = sentiment_plot_df['total'].max()
label_offset = max(max_count * 0.025, 0.15)
for _, row in sentiment_plot_df.iterrows():
    if row['total'] <= 0:
        continue
    ax_counts.text(
        row['year'],
        row['total'] + label_offset,
        str(int(row['total'])),
        ha='center',
        va='bottom',
        fontsize=value_fs,
        color='0.10',
    )
ax_counts.set_ylabel('Number of references', fontsize=label_fs)
ax_counts.tick_params(axis='both', labelsize=tick_fs)
ax_counts.grid(axis='y', color='0.86', linewidth=0.8)
ax_counts.set_axisbelow(True)
ax_counts.spines['top'].set_visible(False)
ax_counts.spines['right'].set_visible(False)
ax_counts.set_ylim(0, max_count + 1)
ax_counts.yaxis.set_major_locator(MaxNLocator(integer=True))
ax_counts.legend(frameon=False, fontsize=legend_fs, loc='upper right')
for target in period_targets:
    start, end = publication_periods[target]
    y_pos = period_y_positions[target]
    start_plot = start - 0.42
    width_plot = end - start + 0.84
    ax_periods.broken_barh(
        [(start_plot, width_plot)],
        (y_pos - period_bar_height / 2, period_bar_height),
        facecolors='0.18',
        edgecolors='0.18',
        linewidth=0.8,
    )
ax_periods.set_yticks([period_y_positions[target] for target in period_targets])
ax_periods.set_yticklabels([format_target(target) for target in period_targets], fontsize=tick_fs)
ax_periods.set_ylim(-period_row_gap, max(period_y_positions.values()) + period_row_gap)
ax_periods.invert_yaxis()
ax_periods.set_xlabel('Year', fontsize=label_fs)
ax_periods.set_ylabel('Publication periods of referenced works', fontsize=label_fs)
ax_periods.tick_params(axis='x', labelsize=tick_fs)
ax_periods.tick_params(axis='y', labelsize=tick_fs)
ax_periods.grid(axis='x', color='0.86', linewidth=0.8)
ax_periods.set_axisbelow(True)
ax_periods.spines['top'].set_visible(False)
ax_periods.spines['right'].set_visible(False)
ax_periods.set_xlim(x_min, x_max)
ax_periods.set_xticks(range(int(x_min), int(x_max) + 1, 5))
fig.tight_layout()
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()


## Post-publication references

In [ ]:
import os
import ast
import matplotlib.pyplot as plt
from collections import defaultdict

novel_publication_years = {
    'sketches_by_boz': (1836, 1836),
    'pickwick_papers': (1836, 1837),
    'oliver_twist': (1837, 1839),
    'nicholas_nickleby': (1838, 1839),
    'old_curiosity_shop': (1840, 1841),
    'barnaby_rudge': (1841, 1841),
    'american_notes': (1842, 1842),
    'a_christmas_carol': (1843, 1843),
    'martin_chuzzlewit': (1843, 1844),
    'dombey_and_son': (1846, 1848),
    'david_copperfield': (1849, 1850),
    'bleak_house': (1852, 1853),
    'hard_times': (1854, 1854),
    'little_dorrit': (1855, 1857),
    'a_tale_of_two_cities': (1859, 1859),
    'great_expectations': (1860, 1861),
    'our_mutual_friend': (1864, 1865),
    'edwin_drood': (1870, 1870),
}
selected_novels = [
    'pickwick_papers',
    'oliver_twist',
    'a_christmas_carol',
    'dombey_and_son',
    'david_copperfield',
    'bleak_house',
]
novel_display_names = {
    'pickwick_papers': 'The Pickwick Papers',
    'oliver_twist': 'Oliver Twist',
    'a_christmas_carol': 'A Christmas Carol',
    'dombey_and_son': 'Dombey and Son',
    'david_copperfield': 'David Copperfield',
    'bleak_house': 'Bleak House',
}


def make_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, (tuple, set)):
        return list(value)
    if isinstance(value, str):
        value = value.strip()
        if not value or value in {'[]', 'None', 'nan'}:
            return []
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
            if isinstance(parsed, (tuple, set)):
                return list(parsed)
        except (ValueError, SyntaxError):
            pass
        return [value]
    return []


def add_year_to_sentences(sentences_df, reviewed_letters):
    df = sentences_df.copy()
    if 'year' in df.columns:
        return df
    year_by_letter_id = {}
    for letter in reviewed_letters:
        letter_id = getattr(letter, 'id', None)
        year = getattr(letter, 'year', None)
        if letter_id is not None:
            year_by_letter_id[letter_id] = year
    df['year'] = df['letter_id'].map(year_by_letter_id)
    return df


def extract_post_publication_data(sentences_df, selected_novels):
    novel_year_data = defaultdict(list)
    for _, row in sentences_df.iterrows():
        year = row.get('year')
        if year is None:
            continue
        try:
            year = int(year)
        except (TypeError, ValueError):
            continue
        target_novels = make_list(row.get('target_novels'))
        for target_novel in target_novels:
            if target_novel not in selected_novels:
                continue
            if target_novel not in novel_publication_years:
                continue
            publication_end = novel_publication_years[target_novel][1]
            if year > publication_end:
                novel_year_data[target_novel].append(year)
    return novel_year_data


def get_publication_label(novel):
    start_year, end_year = novel_publication_years[novel]
    if start_year == end_year:
        return f'Publication year: {start_year}'
    return f'Publication period: {start_year}–{end_year}'


def visualise_novel_data_grid(novel_year_data, selected_novels, output_dir=None):
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    fig, axes = plt.subplots(3, 2, figsize=(14, 15))
    axes = axes.flatten()
    for ax, novel in zip(axes, selected_novels):
        years = novel_year_data.get(novel, [])
        publication_period = novel_publication_years[novel]
        display_name = novel_display_names.get(novel, novel)
        publication_label = get_publication_label(novel)
        if years:
            ax.hist(years, bins=range(1820, 1871), color='black', edgecolor='grey', align='left')
        ax.axvspan(
            publication_period[0],
            publication_period[1],
            color='red',
            alpha=0.3,
            label=publication_label,
        )
        ax.set_title(display_name, fontsize=14)
        ax.set_xlabel('Year', fontsize=12)
        ax.set_ylabel('Number of references', fontsize=12)
        ax.set_xticks(range(1820, 1871, 5))
        ax.set_ylim(0, 90)
        ax.set_yticks(range(0, 91, 10))
        ax.legend(fontsize=9, loc='upper left')
    plt.tight_layout()
    if output_dir:
        output_path = os.path.join(output_dir, 'post_publication_references_selected_novels_grid_3x2.png')
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
sentences_df_with_year = add_year_to_sentences(sentences_df, reviewed_letters)
post_publication_data = extract_post_publication_data(sentences_df_with_year, selected_novels)
for novel in selected_novels:
    print(f'{novel_display_names[novel]}: {len(post_publication_data.get(novel, []))}')
visualise_novel_data_grid(
    post_publication_data,
    selected_novels,
    output_dir=str(paths.FIGURES_DIR / 'post_publication_references'),
)


## Correspondent associations

In [ ]:
import os
import ast
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

OUTPUT_DIR = str(paths.FIGURES_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)
TOP_N_CORRESPONDENTS = 10
TOP_N_WORKS = 10
RESIDUAL_COLOUR_LIMIT = 15
EXCLUDE_CORRESPONDENTS = set()
EXCLUDE_TARGET_WORKS = {'general'}
category_display_names = {
    0: 'Editorial and Publishing Processes',
    1: 'Elements of Fictional Craft',
    2: 'Financial and Legal Aspects',
    3: 'Gratitude and Acknowledgement',
    4: 'Impact and Reception',
    5: 'Inspirations and Influences',
    6: 'Marketing and Promotion',
    7: 'Quality and Evaluation',
    8: 'Writing Process and Creative Insights',
}
category_display_order = [
    'Inspirations and Influences',
    'Writing Process and Creative Insights',
    'Elements of Fictional Craft',
    'Editorial and Publishing Processes',
    'Quality and Evaluation',
    'Financial and Legal Aspects',
    'Gratitude and Acknowledgement',
    'Marketing and Promotion',
    'Impact and Reception',
]
category_axis_labels = {
    'Inspirations and Influences': 'Inspirations\nand Influences',
    'Writing Process and Creative Insights': 'Writing\nProcess',
    'Elements of Fictional Craft': 'Fictional\nCraft',
    'Editorial and Publishing Processes': 'Editorial and\nPublishing',
    'Quality and Evaluation': 'Quality and\nEvaluation',
    'Financial and Legal Aspects': 'Financial and\nLegal',
    'Gratitude and Acknowledgement': 'Gratitude and\nAcknowledgement',
    'Marketing and Promotion': 'Marketing and\nPromotion',
    'Impact and Reception': 'Impact and\nReception',
}
category_name_aliases = {'Writing Process and Creative Insight': 'Writing Process and Creative Insights'}
target_work_display_names = {
    'sketches_by_boz': 'Sketches by Boz',
    'pickwick_papers': 'The Pickwick Papers',
    'oliver_twist': 'Oliver Twist',
    'nicholas_nickleby': 'Nicholas Nickleby',
    'old_curiosity_shop': 'The Old Curiosity Shop',
    'barnaby_rudge': 'Barnaby Rudge',
    'american_notes': 'American Notes',
    'a_christmas_carol': 'A Christmas Carol',
    'martin_chuzzlewit': 'Martin Chuzzlewit',
    'dombey_and_son': 'Dombey and Son',
    'david_copperfield': 'David Copperfield',
    'bleak_house': 'Bleak House',
    'hard_times': 'Hard Times',
    'little_dorrit': 'Little Dorrit',
    'a_tale_of_two_cities': 'A Tale of Two Cities',
    'great_expectations': 'Great Expectations',
    'our_mutual_friend': 'Our Mutual Friend',
    'edwin_drood': 'The Mystery of Edwin Drood',
    'master_humphreys_clock': "Master Humphrey's Clock",
    'the_chimes': 'The Chimes',
    'the_battle_of_life': 'The Battle of Life',
    'the_haunted_man': 'The Haunted Man',
}


def get_value(item, key, default=None):
    if isinstance(item, dict):
        return item.get(key, default)
    return getattr(item, key, default)


def add_correspondent_to_sentences(sentences_df, reviewed_letters):
    df = sentences_df.copy()
    if 'correspondent' in df.columns:
        return df
    if 'norm_recipient' in df.columns:
        df['correspondent'] = df['norm_recipient']
        return df
    if 'recipient' in df.columns:
        df['correspondent'] = df['recipient']
        return df
    correspondent_by_letter_id = {}
    for letter in reviewed_letters:
        letter_id = get_value(letter, 'id')
        correspondent = (
            get_value(letter, 'norm_recipient')
            or get_value(letter, 'recipient')
            or get_value(letter, 'addressee')
        )
        if letter_id is not None:
            correspondent_by_letter_id[letter_id] = correspondent
    df['correspondent'] = df['letter_id'].map(correspondent_by_letter_id)
    return df


def normalise_correspondent_capitalisation(value):
    if value != value.upper():
        return value
    value = value.title()
    small_words = {'And': 'and', 'Of': 'of', 'The': 'the', 'To': 'to', 'In': 'in', 'For': 'for', 'By': 'by'}
    parts = value.split()
    parts = [small_words.get(part, part) for part in parts]
    if parts:
        parts[0] = parts[0][:1].upper() + parts[0][1:]
    return ' '.join(parts)


def clean_correspondent(value):
    if value is None:
        return None
    value = str(value).strip()
    if not value or value.lower() in {'none', 'nan', 'unknown'}:
        return None
    return normalise_correspondent_capitalisation(value)


def clean_category(value):
    if value is None:
        return None
    if isinstance(value, int):
        return category_display_names.get(value)
    if isinstance(value, float) and value.is_integer():
        return category_display_names.get(int(value))
    value = str(value).strip()
    if not value:
        return None
    if value.isdigit():
        return category_display_names.get(int(value))
    return category_name_aliases.get(value, value)


def clean_target_work(value):
    if value is None:
        return None
    value = str(value).strip()
    if not value or value.lower() in {'none', 'nan'}:
        return None
    if value in EXCLUDE_TARGET_WORKS:
        return None
    if value in target_work_display_names:
        return target_work_display_names[value]
    return value.replace('_', ' ').title()


def build_correspondent_category_table(sentences_df):
    counts = defaultdict(lambda: defaultdict(int))
    for _, row in sentences_df.iterrows():
        correspondent = clean_correspondent(row.get('correspondent'))
        if correspondent is None or correspondent in EXCLUDE_CORRESPONDENTS:
            continue
        categories = make_list(row.get('categories'))
        for category in categories:
            category = clean_category(category)
            if category is None:
                continue
            counts[correspondent][category] += 1
    table = pd.DataFrame.from_dict(counts, orient='index').fillna(0).astype(int)
    table = table.reindex(columns=category_display_order, fill_value=0)
    table = table.loc[table.sum(axis=1).sort_values(ascending=False).index]
    return table


def build_correspondent_work_table(sentences_df):
    counts = defaultdict(lambda: defaultdict(int))
    for _, row in sentences_df.iterrows():
        correspondent = clean_correspondent(row.get('correspondent'))
        if correspondent is None or correspondent in EXCLUDE_CORRESPONDENTS:
            continue
        target_works = make_list(row.get('target_novels'))
        for target_work in target_works:
            target_work = clean_target_work(target_work)
            if target_work is None:
                continue
            counts[correspondent][target_work] += 1
    table = pd.DataFrame.from_dict(counts, orient='index').fillna(0).astype(int)
    table = table.loc[table.sum(axis=1).sort_values(ascending=False).index]
    table = table.loc[:, table.sum(axis=0).sort_values(ascending=False).index]
    return table


def expected_counts(table):
    values = table.astype(float)
    grand_total = values.values.sum()
    if grand_total == 0:
        return values * 0
    row_totals = values.sum(axis=1).values.reshape(-1, 1)
    column_totals = values.sum(axis=0).values.reshape(1, -1)
    expected = row_totals @ column_totals / grand_total
    return pd.DataFrame(expected, index=table.index, columns=table.columns)


def standardised_residuals(table):
    observed = table.astype(float)
    expected = expected_counts(observed)
    residuals = (observed - expected) / np.sqrt(expected.replace(0, np.nan))
    residuals = residuals.replace([np.inf, -np.inf], np.nan).fillna(0)
    return residuals


def wrap_labels(labels, width=14):
    return ['\n'.join(textwrap.wrap(str(label), width=width)) for label in labels]


def get_axis_labels(labels, width=14):
    return [
        category_axis_labels.get(label, '\n'.join(textwrap.wrap(str(label), width=width)))
         for label in labels
    ]


def get_soft_diverging_cmap():
    colors = ['#7b3f3f', '#b98282', '#e7d0cc', '#f7f7f5', '#cbdedb', '#7ea8a5', '#3f6f73']
    return LinearSegmentedColormap.from_list('soft_diverging', colors)


def save_figure(fig, base_filename):
    png_path = os.path.join(OUTPUT_DIR, f'{base_filename}.png')
    fig.savefig(png_path, dpi=450, bbox_inches='tight', facecolor='white')


def plot_residual_heatmap(
    table,
    output_basename,
    x_label,
    y_label='Correspondent',
    colourbar_label='Standardised residual',
    figsize=(18, 10),
    x_label_width=14,
    y_label_width=22,
    residual_colour_limit=15,
    x_tick_rotation=45,
):
    max_abs = residual_colour_limit
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    im = ax.imshow(
        table.values,
        aspect='auto',
        cmap=get_soft_diverging_cmap(),
        norm=TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs),
        interpolation='nearest',
    )
    ax.set_xlabel(x_label, fontsize=26, labelpad=20)
    ax.set_ylabel(y_label, fontsize=26, labelpad=18)
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels(
        get_axis_labels(table.columns, width=x_label_width),
        rotation=x_tick_rotation,
        ha='right',
        rotation_mode='anchor',
        fontsize=21,
    )
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels(wrap_labels(table.index, width=y_label_width), fontsize=22)
    ax.set_xticks(np.arange(-0.5, len(table.columns), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(table.index), 1), minor=True)
    ax.grid(which='minor', color='#ffffff', linestyle='-', linewidth=1.2)
    ax.tick_params(which='minor', bottom=False, left=False)
    ax.tick_params(axis='both', which='major', length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.035, extend='both')
    cbar.set_label(colourbar_label, fontsize=22, labelpad=16)
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(labelsize=19)
    for row_index in range(table.shape[0]):
        for col_index in range(table.shape[1]):
            value = float(table.iat[row_index, col_index])
            if abs(value) < 1.5:
                continue
            text_color = '#ffffff' if abs(value) > max_abs * 0.55 else '#1f2a2a'
            ax.text(
                col_index,
                row_index,
                f'{value:.1f}',
                ha='center',
                va='center',
                fontsize=20,
                color=text_color,
            )
    plt.tight_layout()
    save_figure(fig, output_basename)
    plt.show()
sentences_df_with_correspondents = add_correspondent_to_sentences(sentences_df, reviewed_letters)
category_table = build_correspondent_category_table(sentences_df_with_correspondents)
work_table = build_correspondent_work_table(sentences_df_with_correspondents)
top_correspondents = category_table.sum(axis=1).sort_values(ascending=False).head(TOP_N_CORRESPONDENTS).index
top_works = work_table.sum(axis=0).sort_values(ascending=False).head(TOP_N_WORKS).index
category_residuals = standardised_residuals(category_table).loc[top_correspondents, category_display_order]
work_residuals = standardised_residuals(work_table).loc[top_correspondents, top_works]
plot_residual_heatmap(
    category_residuals,
    output_basename='7_correspondent_category_residuals',
    x_label='Reference category',
    figsize=(20, 11.5),
    x_label_width=12,
    y_label_width=22,
    residual_colour_limit=RESIDUAL_COLOUR_LIMIT,
    x_tick_rotation=45,
)
plot_residual_heatmap(
    work_residuals,
    output_basename='7_correspondent_target_residuals',
    x_label='Target work',
    figsize=(20, 10.5),
    x_label_width=14,
    y_label_width=22,
    residual_colour_limit=RESIDUAL_COLOUR_LIMIT,
    x_tick_rotation=45,
)
